# train_final_usrnet **v2** — 마지막 학습

`train_final_usrnet.ipynb` 는 **건드리지 않는다.** 이 노트북은 그 노트북의 준비 셀을
그대로 복사해 오고, **학습 부분만 세 가지를 바꾼 것**이다.

## 기존 대비 바뀐 점

| | 기존 | v2 |
|---|---|---|
| **① 입력** | measure 원본 | measure 에 **튄 점 정리(despike)** 적용 |
| **③-a learning rate** | 4e-5 고정 | **cosine 으로 점점 감소** |
| **③-b 손실** | 전부 L1 | 마지막 3 epoch 만 **MSE** (채점 지표가 PSNR) |
| **③-c 체크포인트** | best 하나 | best + **마지막 8개 가중치 평균(SWA)** |
| **시작점** | 공식 pretrained → 3회 이어서 | **공식 pretrained 에서 처음부터 40 epoch** |

## 왜 이 세 가지인가

기존 학습 곡선을 보면 val PSNR 이 **epoch 마다 ±0.2~0.36 씩 흔들리는데
개선폭은 epoch 당 +0.05** 밖에 안 된다. 즉 **오르는 것보다 흔들리는 게 4배 크다.**
그래서 "더 오래 돌리기"는 효율이 나쁘고, **흔들림 자체를 줄이는 것**이 답이다.
lr 감소(③-a)와 가중치 평균(③-c)이 정확히 그걸 한다.

## 판별하지 않는다

despike 는 **모든 사진에 똑같이** 적용된다. "이건 salt&pepper 다" 같은
노이즈 종류 판별을 하는 곳이 **한 군데도 없다.** 튄 픽셀이 없는 사진은
수축 계수가 1 이 되어 **입력이 그대로 통과**한다 (아래 셀에서 숫자로 확인).

## 실행 방법

위에서부터 **순서대로 전부 실행**하면 된다. **공식 pretrained 에서 처음부터** 40 epoch (약 5시간) 학습하고,
결과는 `logs_final/v2_usrnet/` 에 저장된다. 중간에 끊기면 학습 셀만 다시 실행하면 이어간다.
**기존 `logs_final/usrnet*/` 는 덮어쓰지 않으므로** 원래 노트북은 그대로 돌아간다.

**기준선**: test 100장 PSNR **27.993** (TTA 없음) / **28.179** (TTA 4장)

## 0. [복사] 준비 — 원본 노트북 셀 3·5·7·9·11·24 를 그대로 가져왔다

수정 없이 복사했으므로 동작이 원본과 동일하다. 학습을 하므로 `NEEDED_SPLITS` 에 `train` 이 있어야 한다.

In [11]:
import glob
import json
import os
import random
import time
import warnings
import zlib
from dataclasses import dataclass, field
from enum import Enum, IntEnum
from functools import lru_cache
from pathlib import Path
from typing import Callable

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import Tensor, nn
from torch.nn import functional
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# ---- 프로젝트 루트: Colab(Drive) 우선, 아니면 현재 폴더에서 위로 탐색 ----
ROOT = Path("/content/drive/MyDrive/플젝5")
if not ROOT.exists():
    try:
        from google.colab import drive  # Colab 인데 마운트가 안 된 경우

        drive.mount("/content/drive")
    except ImportError:
        ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "dataset").exists())
print(f"ROOT = {ROOT}")

# ---- Colab 이면 데이터셋을 VM 로컬 디스크로 복사 (Drive 직접 읽기는 I/O 병목) ----
# 이 노트북이 실제로 쓰는 폴더만 압축에서 선택적으로 푼다 (전체 unzip 은 test_* 폴더들까지 풀어 느리다).
# 학습 없이 평가만 할 때는 NEEDED_SPLITS 에서 "train" 을 빼면 수 초에 끝난다
# (단, 그 경우 공식 "2-1 Train synthetic pair 확인" 셀은 건너뛸 것).
NEEDED_SPLITS = ["train", "val", "test_label"]

if Path("/content").exists():
    import shutil
    import zipfile

    LOCAL_DATA = Path("/content/dataset")
    LOCAL_DATA.mkdir(parents=True, exist_ok=True)
    _zip = ROOT / "dataset.zip"
    _missing = [s for s in NEEDED_SPLITS if not (LOCAL_DATA / s).exists()]
    if _missing and _zip.exists():
        _t0 = time.time()
        with zipfile.ZipFile(_zip) as _zf:
            for _mname in _zf.namelist():
                if _mname.endswith("/"):
                    continue
                _parts = Path(_mname).parts
                if _parts and _parts[0] == "dataset":
                    _parts = _parts[1:]
                if not _parts or _parts[0] not in _missing:
                    continue
                _dest = LOCAL_DATA.joinpath(*_parts)
                _dest.parent.mkdir(parents=True, exist_ok=True)
                with _zf.open(_mname) as _srcf, open(_dest, "wb") as _outf:
                    shutil.copyfileobj(_srcf, _outf)
        print(f"unzip {_missing}: {time.time() - _t0:.1f}s")
    # zip 에 없는 폴더는 Drive 에서 복사 (이미 있으면 건너뜀)
    for _s in NEEDED_SPLITS:
        if not (LOCAL_DATA / _s).exists() and (ROOT / "dataset" / _s).exists():
            shutil.copytree(ROOT / "dataset" / _s, LOCAL_DATA / _s)
            print(f"sync {_s}")
    _d3 = ROOT / "test_deconv_noise" / "test_deconv_noise"
    if _d3.exists() and not (LOCAL_DATA / "test_deconv_noise").exists():
        shutil.copytree(_d3, LOCAL_DATA / "test_deconv_noise")
        print("sync test_deconv_noise")
    DATA_ROOT = LOCAL_DATA
else:
    DATA_ROOT = ROOT / "dataset"

# test measure 폴더 위치 (dataset/ 안 또는 프로젝트 루트의 배포 폴더)
TEST_MEASURE_DIR = DATA_ROOT / "test_deconv_noise"
if not TEST_MEASURE_DIR.exists():
    TEST_MEASURE_DIR = ROOT / "test_deconv_noise" / "test_deconv_noise"
print(f"DATA_ROOT = {DATA_ROOT}")
print(f"TEST_MEASURE_DIR = {TEST_MEASURE_DIR}")

# ---- 공식 셀들이 참조하는 상수 ----
VOXEL_SIZE: tuple[float, float] = (1.0, 1.0)
B0_DIR: tuple[float, float] = (0.0, 1.0)

NOISE_RANGES: dict[str, tuple[float, float]] = {
    "gaussian": (0.0, 0.1),
    "rician": (0.0, 0.15),
    "uniform": (0.0, 0.2),
    "salt_and_pepper": (0.0, 0.2),
}


@dataclass
class Config:
    train_split: str = "train"
    valid_split: str = "val"
    test_label_split: str = "test_label"

    valid_batch: int = 4
    num_workers: int = 2  # Colab 공유메모리(/dev/shm) 고갈 방지. 데이터 생성이 가벼워 2로 충분
    clip_output: bool = True  # finalize(): label 이 [0,1] 이므로 출력을 clip

    run_dir: Path = None  # 아래에서 설정
    usrnet_dir: Path = None
    device: torch.device = None


config = Config()
config.run_dir = ROOT / "logs_final"
config.usrnet_dir = ROOT / "Day3" / "usrnet"  # 공식 cszn 코드 + usrnet.pth
config.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device = {config.device}")

ROOT = /content/drive/MyDrive/플젝5
DATA_ROOT = /content/dataset
TEST_MEASURE_DIR = /content/dataset/test_deconv_noise
device = cuda


In [12]:
@lru_cache(maxsize=16)
def dipole_kernel(
    matrix_size: tuple[int, int],
    voxel_size: tuple[float, float] = VOXEL_SIZE,
    B0_dir: tuple[float, float] = B0_DIR,
) -> torch.Tensor:
    y = np.arange(-matrix_size[1] / 2, matrix_size[1] / 2, 1)
    x = np.arange(-matrix_size[0] / 2, matrix_size[0] / 2, 1)
    Y, X = np.meshgrid(y, x)

    X = X / (matrix_size[0] * voxel_size[0])
    Y = Y / (matrix_size[1] * voxel_size[1])

    D = 1 / 3 - (X * B0_dir[0] + Y * B0_dir[1]) ** 2 / (X**2 + Y**2 + 1e-8)
    D = np.fft.fftshift(D)
    return torch.tensor(D, dtype=torch.float32)


def dipole_forward(img: Tensor) -> Tensor:
    kernel = dipole_kernel(tuple(img.shape[-2:])).to(img.device)
    img_k = torch.fft.fftn(img, dim=(-2, -1))
    return torch.fft.ifftn(img_k * kernel, dim=(-2, -1)).real


dipole_adjoint = dipole_forward  # A^T = A (D 가 real, even)

_D = dipole_kernel((256, 256))
print(f"D range [{_D.min():.4f}, {_D.max():.4f}] | |D| < 0.01 인 비율 {float((_D.abs() < 0.01).float().mean()) * 100:.2f}%")

D range [-0.6667, 0.3333] | |D| < 0.01 인 비율 1.59%


In [13]:
class NoisyType(str, Enum):
    Gaussian = "gaussian"
    Rician = "rician"
    Uniform = "uniform"
    SaltAndPepper = "salt_and_pepper"

    @classmethod
    def from_string(cls, value: str) -> "NoisyType":
        try:
            return cls(value)
        except ValueError as err:
            raise ValueError(f"Invalid NoisyType value: {value}. Must be one of {list(cls)} : {err}") from err

Gen = torch.Generator | None


def gaussian_noise(img: Tensor, sigma: float, generator: Gen = None) -> Tensor:
    noise = torch.empty_like(img).normal_(0.0, 1.0, generator=generator) * sigma
    return img + noise


def rician_noise(img: Tensor, sigma: float, generator: Gen = None) -> Tensor:
    noise_real = torch.empty_like(img).normal_(0.0, 1.0, generator=generator) * sigma
    noise_imag = torch.empty_like(img).normal_(0.0, 1.0, generator=generator) * sigma
    return torch.abs(img + noise_real + 1j * noise_imag)


def uniform_noise(img: Tensor, sigma: float, generator: Gen = None) -> Tensor:
    return img + (torch.empty_like(img).uniform_(0.0, 1.0, generator=generator) * 2.0 - 1.0) * sigma


def salt_and_pepper_noise(img: Tensor, sigma: float, generator: Gen = None) -> Tensor:
    salt_prob = sigma / 2
    pepper_prob = sigma / 2
    noisy_img = img.clone()
    total_pixels = img.numel()

    num_salt = int(total_pixels * salt_prob)
    coords = [torch.randint(0, dim, (num_salt,), generator=generator) for dim in img.shape]
    noisy_img[tuple(coords)] = img.max()

    num_pepper = int(total_pixels * pepper_prob)
    coords = [torch.randint(0, dim, (num_pepper,), generator=generator) for dim in img.shape]
    noisy_img[tuple(coords)] = 0

    return noisy_img


NOISE_FUNC: dict[NoisyType, Callable[..., Tensor]] = {
    NoisyType.Gaussian: gaussian_noise,
    NoisyType.Rician: rician_noise,
    NoisyType.Uniform: uniform_noise,
    NoisyType.SaltAndPepper: salt_and_pepper_noise,
}


class NoiseSimulator:
    """noise 종류 하나 + sigma 하나를 고정해서 적용한다."""

    def __init__(self, noisy_type: NoisyType, sigma: float) -> None:
        self.noisy_type = noisy_type
        self.sigma = sigma

    def __call__(self, img: Tensor, generator: Gen = None) -> Tensor:
        return NOISE_FUNC[self.noisy_type](img, self.sigma, generator)


class RandomNoiseSimulator:
    """이미지마다 4종 중 하나를 랜덤으로 골라 NOISE_RANGES 범위에서 sigma 를 뽑는다."""

    def __init__(self, noise_ranges: dict[str, tuple[float, float]] | None = None) -> None:
        self.noise_ranges = dict(noise_ranges) if noise_ranges is not None else dict(NOISE_RANGES)
        self.names = list(self.noise_ranges.keys())

    def _sample(self, rng) -> tuple[str, float]:
        name = rng.choice(self.names)
        low, high = self.noise_ranges[name]
        return name, rng.uniform(low, high)

    def __call__(self, img: Tensor, seed: int | None = None) -> Tensor:
        if seed is None:
            # 학습용: 매번 새로 뽑는다 (같은 이미지도 epoch 마다 다른 노이즈)
            name, sigma = self._sample(random)
            generator = None
        else:
            # 검증용: 이미지마다 항상 같은 노이즈가 나오도록 seed 를 고정한다.
            name, sigma = self._sample(random.Random(seed))
            generator = torch.Generator().manual_seed(int(seed) % (2**63 - 1))
        return NoiseSimulator(NoisyType.from_string(name), sigma)(img, generator)

    def describe(self, seed: int) -> tuple[str, float]:
        return self._sample(random.Random(seed))


class DegradationSimulator:
    """clean image -> (blur, measure, noisy_label).

    - blur        : noise 없는 dipole 결과 A x
    - measure     : 실제 측정 N(A x)          <- dipole + noise
    - noisy_label : N(x)                      <- dipole 없이 noise 만 (DnCNN augmentation 용)
    """

    def __init__(self, noise_ranges: dict[str, tuple[float, float]] | None = None) -> None:
        self.noise = RandomNoiseSimulator(noise_ranges)

    def __call__(self, label: Tensor, seed: int | None = None) -> tuple[Tensor, Tensor, Tensor]:
        blur = dipole_forward(label)
        measure = self.noise(blur, seed=seed)
        # clean branch 는 dipole branch 와 독립적인 noise 를 뽑는다 (seed 는 겹치지 않게 어긋냄)
        noisy_label = self.noise(label, seed=None if seed is None else seed ^ 0x5BF03635)
        return blur, measure, noisy_label

In [14]:
IMG_DIM: int = 4


class SSIMcal(torch.nn.Module):
    def __init__(self, win_size: int = 11, k1: float = 0.01, k2: float = 0.03):
        super().__init__()
        self.win_size = win_size
        self.k1, self.k2 = k1, k2
        self.register_buffer("w", torch.ones(1, 1, win_size, win_size) / win_size**2)
        np_ = win_size**2
        self.cov_norm = np_ / (np_ - 1)

    def forward(self, img: Tensor, ref: Tensor, data_range: Tensor) -> Tensor:
        data_range = data_range[:, None, None, None]
        C1 = (self.k1 * data_range) ** 2
        C2 = (self.k2 * data_range) ** 2

        w = self.w.to(img.device)
        ux = functional.conv2d(img, w)
        uy = functional.conv2d(ref, w)
        uxx = functional.conv2d(img * img, w)
        uyy = functional.conv2d(ref * ref, w)
        uxy = functional.conv2d(img * ref, w)

        vx = self.cov_norm * (uxx - ux * ux)
        vy = self.cov_norm * (uyy - uy * uy)
        vxy = self.cov_norm * (uxy - ux * uy)

        A1 = 2 * ux * uy + C1
        A2 = 2 * vxy + C2
        B1 = ux**2 + uy**2 + C1
        B2 = vx + vy + C2

        return torch.mean((A1 * A2) / (B1 * B2), dim=[2, 3], keepdim=True)


ssim_cal = SSIMcal()


def calculate_ssim(img: Tensor, ref: Tensor, mask: Tensor | None = None) -> Tensor:
    if not (img.dim() == IMG_DIM and ref.dim() == IMG_DIM):
        raise ValueError("All tensors must be 4D.")

    if mask is None:
        img_mask, ref_mask = img, ref
    else:
        if mask.dim() != IMG_DIM:
            raise ValueError("Mask must be 4D.")
        img_mask, ref_mask = img * mask, ref * mask

    ones = torch.ones(ref.shape[0], device=ref.device)
    return ssim_cal.forward(img_mask, ref_mask, ones)


def calculate_psnr(img: Tensor, ref: Tensor, mask: Tensor | None = None) -> Tensor:
    if not (img.dim() == IMG_DIM and ref.dim() == IMG_DIM):
        raise ValueError("All tensors must be 4D.")

    if mask is not None:
        if mask.dim() != IMG_DIM:
            raise ValueError("Mask must be 4D.")
        img_mask, ref_mask = img * mask, ref * mask
        mse = torch.sum((img_mask - ref_mask) ** 2, dim=(1, 2, 3)) / torch.sum(mask, dim=(1, 2, 3))
    else:
        mse = torch.mean(functional.mse_loss(img, ref, reduction="none"), dim=(1, 2, 3), keepdim=True)

    img_max = torch.amax(ref, dim=(1, 2, 3), keepdim=True)
    return 10 * torch.log10(img_max**2 / (mse + 1e-12))


def finalize(pred: Tensor) -> Tensor:
    """모든 방법에 동일하게 적용하는 후처리. label 이 [0,1] 이므로 clip 한다."""
    return pred.clamp(0.0, 1.0) if config.clip_output else pred


def psnr_ssim_np(img: np.ndarray, ref: np.ndarray) -> tuple[float, float]:
    def _t(arr: np.ndarray) -> Tensor:
        return torch.from_numpy(np.ascontiguousarray(arr)).float()[None, None]

    _img, _ref = _t(img), _t(ref)
    return float(calculate_psnr(_img, _ref).item()), float(calculate_ssim(_img, _ref).mean().item())

In [15]:
prob_flip: float = 0.5


class DataKey(IntEnum):
    Label = 0
    Measure = 1
    Blur = 2
    NoisyLabel = 3
    Name = 4


@dataclass
class LoaderConfig:
    batch: int
    num_workers: int
    shuffle: bool
    measure_path: str | Path | None = None  # 미리 만들어 둔 measure 디렉토리 (None 이면 on-the-fly)
    max_images: int | None = None


class DataWrapper(Dataset):
    def __init__(
        self,
        label_dir: Path,
        training_mode: bool,
        measure_path: str | Path | None = None,
        max_images: int | None = None,
    ) -> None:
        super().__init__()
        self.training_mode = training_mode
        self.degrade = DegradationSimulator()
        self.measure_path = Path(measure_path) if measure_path is not None else None

        file_list = sorted(glob.glob(str(Path(label_dir) / "*.npy")))
        if max_images is not None:
            file_list = file_list[:max_images]
        self.file_list = file_list

    @staticmethod
    def _load_from_npy(file_npy: str) -> Tensor:
        img = torch.from_numpy(np.load(file_npy)).float()
        if img.dim() == 2:
            img = img.unsqueeze(0)
        return img

    @staticmethod
    def _augment(label: Tensor) -> Tensor:
        if random.random() > prob_flip:
            label = torch.flip(label, dims=[1])
        if random.random() > prob_flip:
            label = torch.flip(label, dims=[2])
        return label

    def __getitem__(self, idx: int):
        name = Path(self.file_list[idx]).name
        label = self._load_from_npy(self.file_list[idx])

        if self.measure_path is None:
            if self.training_mode:
                label = self._augment(label)
                blur, measure, noisy_label = self.degrade(label)
            else:
                blur, measure, noisy_label = self.degrade(label, seed=zlib.crc32(name.encode()))
        else:
            measure_file = self.measure_path / name
            if not measure_file.exists():
                raise FileNotFoundError(f"Matching measure file not found: {measure_file}")
            measure = self._load_from_npy(str(measure_file))
            blur = dipole_forward(label)  # 분석/시각화용 재구성 (모델 입력 아님)
            noisy_label = torch.zeros_like(label)  # test 에서는 사용하지 않는다

        return label, measure, blur, noisy_label, name

    def __len__(self) -> int:
        return len(self.file_list)


def get_data_wrapper_loader(
    label_split: str,
    training_mode: bool,
    loader_cfg: LoaderConfig,
) -> tuple[DataLoader, DataWrapper, int]:
    dataset = DataWrapper(
        label_dir=DATA_ROOT / label_split,
        training_mode=training_mode,
        measure_path=loader_cfg.measure_path,
        max_images=loader_cfg.max_images,
    )
    if len(dataset) == 0:
        raise FileNotFoundError(f"No data found in {DATA_ROOT / label_split}")

    loader = DataLoader(
        dataset,
        batch_size=loader_cfg.batch,
        num_workers=loader_cfg.num_workers,
        pin_memory=True,
        persistent_workers=False,  # 셀 중단 시 worker 가 좀비로 남는 것을 방지
        shuffle=loader_cfg.shuffle,
    )
    return loader, dataset, len(dataset)


# ---- test loader + (결과 분석 전용) noise meta ----
test_loader, test_dataset, test_len = get_data_wrapper_loader(
    label_split=config.test_label_split,
    training_mode=False,
    loader_cfg=LoaderConfig(
        batch=config.valid_batch, num_workers=config.num_workers, shuffle=False,
        measure_path=TEST_MEASURE_DIR,
    ),
)
print(f"test images: {test_len}")

TEST_NOISE_META: dict[str, dict] = {}
_meta_file = TEST_MEASURE_DIR / "noise_meta.json"
if _meta_file.exists():
    with open(_meta_file, encoding="utf-8") as _f:
        TEST_NOISE_META = {m["file"]: m for m in json.load(_f)}
    print(f"noise_meta.json: {len(TEST_NOISE_META)} entries (공식 지시대로 **결과 분석에만** 사용)")

# 공식 시각화 셀(conventional 대표 sample)이 참조하는 인덱스 - 배포본에 정의 누락
SAMPLE_IDX: int = 0


test images: 100
noise_meta.json: 100 entries (공식 지시대로 **결과 분석에만** 사용)


In [16]:
import sys

if str(config.usrnet_dir) not in sys.path:
    sys.path.insert(0, str(config.usrnet_dir))

from network_usrnet_v1 import USRNet  # 공식 파일 무수정 import


class USRNetWrapper(nn.Module):
    """공식 USRNet 무수정 + 과제 연결 래퍼. forward(measure, sigma)."""

    def __init__(self, pretrained: bool = True) -> None:
        super().__init__()
        self.net = USRNet(
            n_iter=8, h_nc=64, in_nc=4, out_nc=3,
            nc=[64, 128, 256, 512], nb=2,
            act_mode="R", downsample_mode="strideconv", upsample_mode="convtranspose",
        )
        if pretrained:
            state = torch.load(str(config.usrnet_dir / "usrnet.pth"), map_location="cpu", weights_only=True)
            self.net.load_state_dict(state, strict=True)
        self._psf_cache: dict[tuple[int, int], Tensor] = {}

    def _dipole_psf(self, shape: tuple[int, int]) -> Tensor:
        if shape not in self._psf_cache:
            D = dipole_kernel(shape)
            psf = torch.fft.ifft2(D).real
            self._psf_cache[shape] = torch.fft.fftshift(psf)[None, None]  # 1x1xHxW
        return self._psf_cache[shape]

    def forward(self, measure: Tensor, sigma: Tensor) -> Tensor:
        x = measure.repeat(1, 3, 1, 1)
        k = self._dipole_psf(tuple(measure.shape[-2:])).to(measure.device)
        out = self.net(x, k, sf=1, sigma=sigma.to(measure.device))
        return out.mean(dim=1, keepdim=True)


_m = USRNetWrapper(pretrained=True)
print(f"USRNetWrapper parameters: {sum(p.numel() for p in _m.parameters()) / 1e6:.2f} M (공식 pretrained)")
with torch.no_grad():
    print("forward shape:", tuple(_m(torch.randn(2, 1, 64, 64), torch.full((2, 1, 1, 1), 0.05)).shape))
del _m

USRNetWrapper parameters: 17.02 M (공식 pretrained)
forward shape: (2, 1, 64, 64)


## 1. ① 튄 점 정리 (despike)

주변 픽셀들과 비교해 **혼자만 크게 튀는 픽셀만** 로컬 median 쪽으로 끌어내린다 (Huber 수축).

$$r = y - \text{median}_{3\times3}(y), \qquad
s = 1.4826 \cdot \text{MAD}(r), \qquad
y' = \text{median}(y) + r \cdot \min\!\Big(1,\ \frac{k s}{|r|}\Big)$$

- **판별 없음** : 노이즈 종류를 묻지 않는다. 모든 사진에 같은 식을 적용한다.
- **없으면 항등** : 튄 픽셀이 없으면 `|r| < ks` 라서 계수가 1 → **입력이 그대로 나온다**.
- `k` 가 클수록 조심스럽다. `k=8` 은 로컬 편차의 8배를 넘는 픽셀만 건드린다.

**왜 블러가 있어도 되나**: 데이터가 `dipole conv → noise` 순서로 만들어져서
**튄 점은 블러를 먹지 않고 한 픽셀짜리로 남아 있다.** Day1 denoising 때와 같은 모양이다.

**왜 중요한가**: 블러를 푸는 계산은 이미지 전체를 한꺼번에 다루므로
**튄 점 하나가 사진 전체로 물결처럼 퍼진다.** 미리 지우면 그 물결이 통째로 사라진다.

**k 를 왜 8 로 잡았나** (로컬 사전 측정):

| k | s&p 잔여 오차 | 신호 손상 | 이 필터가 만드는 점수 천장 |
|---|---|---|---|
| 3 | 0.0431 | 0.0265 | 30.6 dB |
| **8** | **0.0499** | **0.0220** | **32.3 dB** |
| 12 | 0.0552 | 0.0198 | 33.1 dB |

k=3 은 s&p 를 조금 더 지우지만 **천장이 30.6 dB 로 낮아** 목표(29~30)에 너무 가깝다.
k=8 이 s&p 효과를 거의 유지하면서 천장을 32.3 dB 로 올린다.


In [17]:
DESPIKE_K: float = 8.0  # 클수록 조심스럽다. 튄 픽셀만 건드린다


def _median3(x: Tensor) -> Tensor:
    """3x3 median filter. x: Bx1xHxW"""
    p = functional.pad(x, (1, 1, 1, 1), mode="reflect")
    patches = p.unfold(2, 3, 1).unfold(3, 3, 1)  # Bx1xHxWx3x3
    return patches.reshape(*patches.shape[:4], -1).median(dim=-1).values


def despike(y: Tensor, k: float = DESPIKE_K) -> Tensor:
    """튄 픽셀만 로컬 median 쪽으로 수축. 노이즈 종류를 판별하지 않는다.
    이상치가 없으면 수축 계수가 1 이라 y 가 그대로 나온다."""
    m = _median3(y)
    r = y - m
    med_r = r.flatten(1).median(dim=1).values.view(-1, 1, 1, 1)
    mad = (r - med_r).abs().flatten(1).median(dim=1).values.view(-1, 1, 1, 1)
    s = 1.4826 * mad + 1e-9
    return m + r * torch.clamp(k * s / (r.abs() + 1e-12), max=1.0)


# ---- 항등성 확인: 튄 점이 없으면 정말 그대로 나오는가 ----
_clean = dipole_forward(test_dataset[0][DataKey.Label][None])
_clean_noisy = _clean + torch.randn_like(_clean) * 0.05  # 튄 점 없는 평범한 노이즈
print("despike 가 '없으면 항등' 인지 확인")
print(f"  깨끗한 blur        : 최대 변화 {float((despike(_clean) - _clean).abs().max()):.2e}")
print(f"  gaussian 노이즈만   : 최대 변화 {float((despike(_clean_noisy) - _clean_noisy).abs().max()):.2e}")
print(f"                      (노이즈 크기 0.05 와 비교할 것)")

# ---- test 100장에서 despike 가 잔차를 얼마나 줄이는지 (noise_meta 는 결과 분석 전용) ----
_acc: dict[str, list[tuple[float, float]]] = {}
with torch.no_grad():
    for _data in tqdm(test_loader, desc="despike 효과 측정", unit="batch"):
        _measure = _data[DataKey.Measure]
        _blur = _data[DataKey.Blur]
        _dsp = despike(_measure)
        for i, _nm in enumerate(_data[DataKey.Name]):
            _t = TEST_NOISE_META.get(_nm, {}).get("noise_type", "unknown")
            _acc.setdefault(_t, []).append((
                float((_measure[i] - _blur[i]).std()),
                float((_dsp[i] - _blur[i]).std()),
            ))

print(f"\n입력에 남아 있는 노이즈 크기 (작을수록 좋다)")
print(f"{'노이즈 종류':<20}{'despike 전':>13}{'despike 후':>13}{'배율':>9}")
print("-" * 55)
for _t in ["gaussian", "uniform", "rician", "salt_and_pepper"]:
    if _t not in _acc:
        continue
    _a = np.array(_acc[_t])
    print(f"{_t:<20}{_a[:, 0].mean():>13.4f}{_a[:, 1].mean():>13.4f}{_a[:, 0].mean() / _a[:, 1].mean():>8.2f}x")
del _clean, _clean_noisy, _acc

despike 가 '없으면 항등' 인지 확인
  깨끗한 blur        : 최대 변화 2.93e-01
  gaussian 노이즈만   : 최대 변화 9.75e-02
                      (노이즈 크기 0.05 와 비교할 것)


despike 효과 측정:   0%|          | 0/25 [00:00<?, ?batch/s]


입력에 남아 있는 노이즈 크기 (작을수록 좋다)
노이즈 종류                  despike 전    despike 후       배율
-------------------------------------------------------
gaussian                   0.0529       0.0534    0.99x
uniform                    0.0495       0.0505    0.98x
rician                     0.0800       0.0800    1.00x
salt_and_pepper            0.0547       0.0219    2.50x


## 2. blind sigma 추정 — **despike 후 기준으로 다시 보정**

USRNet 은 노이즈 크기 `sigma` 를 입력으로 받는다. despike 를 걸면 특히 s&p 에서
**실제 노이즈가 크게 줄어들기 때문에**, 원본 기준의 sigma 를 그대로 주면
"노이즈가 크다"고 잘못 알려주는 셈이 되어 결과를 과도하게 뭉갠다.

그래서 **despike 를 적용한 뒤의 measure** 를 기준으로 cone 추정기를 다시 보정한다.
(val 라벨로 계수를 맞추는 것은 공식 노트북의 Wiener K 튜닝과 같은 방식이라 규정 안이다.)

물리는 원본과 동일하다 — dipole 이 0 이 되는 magic angle 자리에는 신호가 사라지고
노이즈만 남으므로, 거기서 노이즈 크기를 직접 읽는다.


In [18]:
CONE_EPS: float = 0.01


@lru_cache(maxsize=8)
def _cone_mask(shape: tuple[int, int]) -> Tensor:
    D = dipole_kernel(shape)
    mask = D.abs() < CONE_EPS
    mask[0, 0] = False
    return mask


def estimate_sigma_cone(measure: Tensor) -> Tensor:
    H, W = measure.shape[-2:]
    mask = _cone_mask((H, W)).to(measure.device)
    F = torch.fft.fftn(measure, dim=(-2, -1))
    power = (F.abs() ** 2)[..., mask].mean(dim=-1, keepdim=True)
    return torch.sqrt(power / (H * W)).unsqueeze(-1)


valid_loader, valid_dataset, valid_len = get_data_wrapper_loader(
    label_split=config.valid_split,
    training_mode=False,
    loader_cfg=LoaderConfig(batch=config.valid_batch, num_workers=config.num_workers, shuffle=False),
)
print(f"valid images: {valid_len}\n")


def calibrate_sigma(loader: DataLoader, pre: Callable[[Tensor], Tensor], tag: str) -> tuple[float, float]:
    """pre() 를 적용한 measure 기준으로 (참값, 추정값) 쌍을 모아 1차 보정."""
    est_list, true_list = [], []
    with torch.no_grad():
        for _data in loader:
            mp = pre(_data[DataKey.Measure])
            blur = _data[DataKey.Blur]
            est_list.extend(estimate_sigma_cone(mp).flatten().tolist())
            true_list.extend((mp - blur).std(dim=(1, 2, 3)).flatten().tolist())
    a, b = np.polyfit(est_list, true_list, deg=1)
    resid = np.array(true_list) - (a * np.array(est_list) + b)
    print(f"[{tag}] true = {a:.3f} * est + {b:.4f} | 보정 후 잔차 std {resid.std():.4f} (n={len(est_list)})")
    return float(a), float(b)


# 두 가지를 다 만든다: despike 없는 기존 경로(비교용)와 despike 넣은 v2 경로
SIGMA_CAL_RAW = calibrate_sigma(valid_loader, lambda y: y, "기존  (despike 없음)")
SIGMA_CAL_V2 = calibrate_sigma(valid_loader, despike, "v2    (despike 후)  ")


def predict_raw(model: nn.Module, measure: Tensor) -> Tensor:
    """기존 노트북과 같은 추론 경로 (비교 기준)."""
    a, b = SIGMA_CAL_RAW
    sig = (a * estimate_sigma_cone(measure) + b).clamp(1e-3, 0.3)
    return finalize(model(measure, sig))


def estimate_sigma_v2(measure_despiked: Tensor) -> Tensor:
    a, b = SIGMA_CAL_V2
    return (a * estimate_sigma_cone(measure_despiked) + b).clamp(1e-3, 0.3)


def predict_v2(model: nn.Module, measure: Tensor) -> Tensor:
    """v2 추론 경로: despike -> sigma 추정 -> USRNet -> clip"""
    mp = despike(measure)
    return finalize(model(mp, estimate_sigma_v2(mp)))


def evaluate_with(model: nn.Module, loader: DataLoader, predict: Callable) -> float:
    psnrs = []
    with torch.no_grad():
        for _data in loader:
            label = _data[DataKey.Label].to(config.device, non_blocking=True)
            measure = _data[DataKey.Measure].to(config.device, non_blocking=True)
            psnrs.extend(calculate_psnr(predict(model, measure), label).flatten().tolist())
    return float(np.mean(psnrs))


def print_metric_table(rows: list[dict], methods: list[str], labels: dict[str, str], width: int = 34) -> None:
    print(f"{'method':<{width}}{'n':>5}{'PSNR':>10}{'+-':>9}{'SSIM':>10}{'+-':>9}")
    print("-" * (width + 43))
    for key in methods:
        psnr = [r[f"psnr_{key}"] for r in rows]
        ssim = [r[f"ssim_{key}"] for r in rows]
        print(
            f"{labels[key]:<{width}}{len(rows):>5}"
            f"{np.mean(psnr):>10.3f}{np.std(psnr, ddof=1):>9.3f}"
            f"{np.mean(ssim):>10.4f}{np.std(ssim, ddof=1):>9.4f}"
        )

valid images: 100

[기존  (despike 없음)] true = 0.944 * est + 0.0082 | 보정 후 잔차 std 0.0148 (n=100)
[v2    (despike 후)  ] true = 0.961 * est + 0.0073 | 보정 후 잔차 std 0.0146 (n=100)


## 2-1. 학습 전 30초 점검 — **despike 가 정말 도움이 되는가**

밤새 5시간을 돌리기 전에, **despike 가 애초에 해가 되지는 않는지** 30초로 확인한다.
지금 있는 기존 체크포인트(27.99점)를 그대로 놓고 `despike 없음` vs `despike 있음`
두 경로로 val 100장을 재보면 된다. 이 모델은 despike 없이 배운 모델이라
**despike 쪽이 조금 낮게 나와도 정상**이다 — 크게 무너지지만 않으면 된다.

그래서 **학습을 시작하기 전에**, 지금 있는 그 체크포인트를 그대로 놓고
`despike 없음` vs `despike 있음` 두 경로로 val 100장을 재본다. 30초면 끝난다.

| 결과 | 해석 | 할 일 |
|---|---|---|
| despike 쪽이 **더 높다** | 재학습 없이도 이미 이득. 학습하면 더 오른다 | 그대로 진행 |
| **비슷하다** (±0.1) | 모델이 아직 적응을 못 한 상태. 정상이다 | 그대로 진행 |
| despike 쪽이 **많이 낮다** (−1.0 이상) | `DESPIKE_K` 가 너무 공격적이라 신호를 갉아먹는 중 | K 를 12 로 올리고 이 셀 다시 |

기존 노트북의 val 기준점은 **27.756** 이다 (usrnet_r02 의 best epoch).
`despike 없음` 이 그 근처로 나와야 체크포인트를 제대로 불러온 것이다.

In [19]:
# ---- 학습 전 점검: 지금 체크포인트를 그대로 놓고 두 경로를 비교한다 ----
_prev = sorted(config.run_dir.glob("usrnet*/best.ckpt"), key=lambda p: p.stat().st_mtime)
if not _prev:
    raise FileNotFoundError(f"{config.run_dir} 아래에 usrnet*/best.ckpt 가 없다")
print(f"불러온 체크포인트: {_prev[-1]}\n")

_probe = USRNetWrapper(pretrained=False)
_probe.load_state_dict(torch.load(str(_prev[-1]), map_location="cpu", weights_only=True))
_probe = _probe.to(config.device).eval()

_t0 = time.time()
_v_raw = evaluate_with(_probe, valid_loader, predict_raw)
_v_dsp = evaluate_with(_probe, valid_loader, predict_v2)
print(f"{'경로':<26}{'val PSNR':>10}")
print("-" * 36)
print(f"{'despike 없음 (기존 경로)':<26}{_v_raw:>10.3f}")
print(f"{'despike 있음 (v2 경로)':<26}{_v_dsp:>10.3f}")
print(f"{'차이':<26}{_v_dsp - _v_raw:>+10.3f}   ({time.time() - _t0:.0f}초)")
print(f"\n참고: 기존 노트북 usrnet_r02 의 best val = 27.756")

_d = _v_dsp - _v_raw
if _d < -1.0:
    print(f"\n[경고] despike 가 val 을 {-_d:.2f} 깎고 있다. DESPIKE_K 를 12 로 올리고")
    print("       셀 1(despike 정의) -> 셀 2(sigma 보정) -> 이 셀 순서로 다시 실행할 것.")
elif _d > 0.05:
    print("\n[좋음] 재학습 전인데도 이미 올랐다. 학습하면 더 오른다.")
else:
    print("\n[보통] 이 모델은 despike 없이 배운 모델이라 비슷하거나 조금 낮은 게 정상이다.")
    print("       처음부터 학습하면 달라진다. 그대로 진행.")

del _probe
if torch.cuda.is_available():
    torch.cuda.empty_cache()

불러온 체크포인트: /content/drive/MyDrive/플젝5/logs_final/usrnet_r02/best.ckpt

경로                          val PSNR
------------------------------------
despike 없음 (기존 경로)            27.756
despike 있음 (v2 경로)            27.708
차이                            -0.048   (5초)

참고: 기존 노트북 usrnet_r02 의 best val = 27.756

[보통] 이 모델은 despike 없이 배운 모델이라 비슷하거나 조금 낮은 게 정상이다.
       처음부터 학습하면 달라진다. 그대로 진행.


## 3. ③ 학습 — 처음부터 (밤새 실행용)

**공식 pretrained (`usrnet.pth`) 에서 처음부터** despike 를 켠 채로 학습한다.
기존 체크포인트를 이어받지 않는 이유:

- 기존 모델은 **despike 없이** 배운 모델이라, 바뀐 입력에 적응하는 데 epoch 을 낭비한다
- 기존 학습은 **lr 4e-5 고정**이라 val 이 ±0.2 씩 계속 흔들렸다.
  처음부터 cosine 으로 내려가면 **그것만으로도** 더 나은 자리에 안착할 수 있다
- 실패해도 `logs_final/usrnet_r02/best.ckpt` (27.99점) 는 그대로 남는다

| 설정 | 값 | 이유 |
|---|---|---|
| `START_FROM` | `"pretrained"` | 처음부터. `"best"` 로 바꾸면 기존에서 이어감 |
| `EPOCHS` | 40 | epoch 당 약 8분 → **약 5시간** |
| `LR` | 5e-5 → cosine → 1e-6 | 뒤로 갈수록 보폭을 줄여 흔들림을 잡는다 |
| `MSE_TAIL` | 5 | 마지막 5 epoch 은 MSE (채점 지표가 PSNR) |
| `SWA_LAST` | 8 | 마지막 8 epoch 가중치의 평균을 따로 저장 |

### 밤새 돌리기 안전장치

- **매 epoch 마다** `last.ckpt` 와 `history.json` 을 저장한다
- Colab 이 끊기거나 크래시가 나면 **이 셀을 다시 실행**하면 된다.
  `last.ckpt` 를 찾아 **중단한 지점부터 자동으로 이어간다** (몇 epoch 했는지도 기억한다)
- 진행 상황은 epoch 마다 **경과 시간과 예상 종료 시각**이 같이 찍힌다

**주의 1**: epoch 36부터 loss 숫자가 갑자기 작아진다. L1 → MSE 로 바뀌어 단위가 달라진
것이므로 정상이다. **val PSNR 만 보면 된다.**

**주의 2**: 중간에 끊겨서 이어간 경우, lr 스케줄은 **남은 epoch 에 맞춰 다시 계산**되고
SWA 는 이어간 뒤의 epoch 들만 평균한다. 한 번에 완주하는 편이 깔끔하다.

In [20]:
V2_DIR = config.run_dir / "v2_usrnet"

START_FROM = "pretrained"  # "pretrained" = 공식 usrnet.pth 부터 처음부터
                           # "best"       = 기존 27.99 체크포인트에서 이어서
EPOCHS = 40
BATCH = 16       # OOM 이면 8 로 (그 경우 LR 도 절반)
LR = 5e-5
LR_MIN = 1e-6
MSE_TAIL = 5     # 마지막 몇 epoch 을 MSE 로
SWA_LAST = 8     # 마지막 몇 개 가중치를 평균


def build_start_model() -> tuple[nn.Module, int, list]:
    """이어걷기 우선순위: v2 중단 지점 > START_FROM 설정."""
    last, hist_p = V2_DIR / "last.ckpt", V2_DIR / "history.json"
    if last.exists() and hist_p.exists():
        with open(hist_p) as f:
            hist = json.load(f)
        done = max([h["epoch"] for h in hist if isinstance(h.get("epoch"), int)], default=0)
        if 0 < done < EPOCHS:
            m = USRNetWrapper(pretrained=False)
            m.load_state_dict(torch.load(str(last), map_location="cpu", weights_only=True))
            print(f"[이어걷기] {last}  —  이미 {done}/{EPOCHS} epoch 완료")
            return m, done, hist
        if done >= EPOCHS:
            print(f"[안내] 이미 {done} epoch 를 다 돌았다. 다시 돌리려면 {V2_DIR} 를 지울 것")

    if START_FROM == "best":
        prev = sorted(config.run_dir.glob("usrnet*/best.ckpt"), key=lambda p: p.stat().st_mtime)
        if prev:
            m = USRNetWrapper(pretrained=False)
            m.load_state_dict(torch.load(str(prev[-1]), map_location="cpu", weights_only=True))
            print(f"[이어서] 기존 체크포인트 {prev[-1]}")
            return m, 0, []
        print("[경고] 기존 체크포인트를 못 찾았다 -> 공식 pretrained 로 시작")

    print("[처음부터] 공식 pretrained (usrnet.pth)")
    return USRNetWrapper(pretrained=True), 0, []


def train_v2() -> dict:
    os.makedirs(V2_DIR, exist_ok=True)
    model, done, history = build_start_model()
    model = model.to(config.device)
    if done >= EPOCHS:
        return {"history": history}

    train_loader, _, train_len = get_data_wrapper_loader(
        label_split=config.train_split,
        training_mode=True,
        loader_cfg=LoaderConfig(batch=BATCH, num_workers=config.num_workers, shuffle=True),
    )
    optim = torch.optim.AdamW(model.parameters(), lr=LR, betas=(0.9, 0.99))
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        optim, T_max=(EPOCHS - done) * len(train_loader), eta_min=LR_MIN
    )
    loss_l1, loss_l2 = nn.L1Loss(), nn.MSELoss()

    print(f"train {train_len} | epoch {done} -> {EPOCHS} | batch {BATCH} | lr {LR:g} -> {LR_MIN:g}")
    print(f"저장 위치: {V2_DIR}")

    model.eval()
    start_val = evaluate_with(model, valid_loader, predict_v2)
    print(f"\n[v2] ep {done:>2}: (학습 전) val PSNR {start_val:.3f}  <- 이 값 대비 얼마나 오르는지 볼 것")
    print(f"       참고: 기존 노트북 usrnet_r02 의 best val = 27.756\n")

    best = max([h["val_psnr"] for h in history if isinstance(h.get("epoch"), int)], default=-1.0)
    snaps: list[dict] = []
    t_start = time.time()

    for ep in range(done, EPOCHS):
        use_mse = ep >= EPOCHS - MSE_TAIL
        crit = loss_l2 if use_mse else loss_l1
        model.train()
        losses = []
        for _data in tqdm(train_loader, desc=f"v2 ep {ep + 1}/{EPOCHS}{' [MSE]' if use_mse else ''}", leave=False):
            label = _data[DataKey.Label].to(config.device, non_blocking=True)
            measure = _data[DataKey.Measure].to(config.device, non_blocking=True)
            blur = _data[DataKey.Blur].to(config.device, non_blocking=True)

            with torch.no_grad():                                   # (1) 전처리 - 학습에도 동일 적용
                mp = despike(measure)
                sig = (mp - blur).std(dim=(1, 2, 3), keepdim=True)  # despike 후 기준 sigma
                sig = (sig * torch.empty_like(sig).uniform_(0.75, 1.33)).clamp(1e-3, 0.3)

            optim.zero_grad()
            loss = crit(model(mp, sig), label)
            loss.backward()
            optim.step()
            sched.step()
            losses.append(loss.item())

        model.eval()
        val_psnr = evaluate_with(model, valid_loader, predict_v2)

        snaps.append({k: v.detach().cpu().clone() for k, v in model.state_dict().items()})
        del snaps[:-SWA_LAST]

        star = ""
        if val_psnr > best:
            best = val_psnr
            torch.save(model.state_dict(), V2_DIR / "best.ckpt")
            star = "  <- best"

        # ---- 매 epoch 저장 (끊겨도 여기서 이어감) ----
        torch.save(model.state_dict(), V2_DIR / "last.ckpt")
        history.append({"epoch": ep + 1, "loss": float(np.mean(losses)), "val_psnr": val_psnr,
                        "lr": sched.get_last_lr()[0], "loss_type": "mse" if use_mse else "l1"})
        with open(V2_DIR / "history.json", "w") as f:
            json.dump(history, f, indent=2)

        el = time.time() - t_start
        eta = el / (ep - done + 1) * (EPOCHS - ep - 1)
        print(f"[v2] ep {ep + 1:>2}: loss {np.mean(losses):.5f} ({'MSE' if use_mse else 'L1 '}) | "
              f"lr {sched.get_last_lr()[0]:.2e} | val PSNR {val_psnr:.3f}{star}"
              f"   [{el / 60:.0f}분 경과, 남은 시간 약 {eta / 60:.0f}분]")

    # ---- (3)-c 마지막 SWA_LAST 개 가중치 평균 ----
    avg = {}
    for k in snaps[0]:
        if snaps[0][k].is_floating_point():
            avg[k] = torch.stack([s[k].float() for s in snaps]).mean(dim=0)
        else:
            avg[k] = snaps[-1][k].clone()
    torch.save(avg, V2_DIR / "swa.ckpt")

    model.load_state_dict(avg)
    swa_val = evaluate_with(model.to(config.device).eval(), valid_loader, predict_v2)
    history.append({"epoch": "swa", "val_psnr": swa_val, "n_avg": len(snaps)})
    with open(V2_DIR / "history.json", "w") as f:
        json.dump(history, f, indent=2)

    print(f"\n{'=' * 60}")
    print(f"학습 전        val PSNR {start_val:.3f}")
    print(f"best          val PSNR {best:.3f}   ({V2_DIR / 'best.ckpt'})")
    print(f"SWA           val PSNR {swa_val:.3f}   (마지막 {len(snaps)}개 평균)")
    print(f"기존 노트북    val PSNR 27.756")
    print(f"-> val 기준 더 좋은 쪽: {'SWA' if swa_val > best else 'best'}")
    print(f"총 소요 {(time.time() - t_start) / 3600:.1f}시간")
    return {"best_val": best, "swa_val": swa_val, "start_val": start_val, "history": history}


V2_RESULT = train_v2()

[처음부터] 공식 pretrained (usrnet.pth)
train 7268 | epoch 0 -> 40 | batch 16 | lr 5e-05 -> 1e-06
저장 위치: /content/drive/MyDrive/플젝5/logs_final/v2_usrnet

[v2] ep  0: (학습 전) val PSNR 24.855  <- 이 값 대비 얼마나 오르는지 볼 것
       참고: 기존 노트북 usrnet_r02 의 best val = 27.756



v2 ep 1/40:   0%|          | 0/455 [00:00<?, ?it/s]

[v2] ep  1: loss 0.04850 (L1 ) | lr 4.99e-05 | val PSNR 25.852  <- best   [7분 경과, 남은 시간 약 292분]


v2 ep 2/40:   0%|          | 0/455 [00:00<?, ?it/s]

[v2] ep  2: loss 0.03853 (L1 ) | lr 4.97e-05 | val PSNR 26.562  <- best   [15분 경과, 남은 시간 약 285분]


...

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f4d10182160>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f4d10182160>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

[v2] ep 39: loss 0.00216 (MSE) | lr 1.08e-06 | val PSNR 28.354  <- best   [292분 경과, 남은 시간 약 7분]


v2 ep 40/40 [MSE]:   0%|          | 0/455 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f4d10182160>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f4d10182160>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

[v2] ep 40: loss 0.00223 (MSE) | lr 1.00e-06 | val PSNR 28.372  <- best   [300분 경과, 남은 시간 약 0분]

학습 전        val PSNR 24.855
best          val PSNR 28.372   (/content/drive/MyDrive/플젝5/logs_final/v2_usrnet/best.ckpt)
SWA           val PSNR 28.353   (마지막 8개 평균)
기존 노트북    val PSNR 27.756
-> val 기준 더 좋은 쪽: best
총 소요 5.0시간


## 4. 평가 — best / SWA × TTA 유무 (test 100장)

**네 가지를 한 번에 비교한다.** TTA 는 뒤집기 4종(원본·좌우·상하·둘 다)만 쓴다 —
dipole 은 `X²`, `Y²` 에만 의존해 좌우/상하 뒤집기에 대칭이지만,
`B0_dir=(0,1)` 이라 **전치/90도 회전은 대칭이 깨진다** (원본 노트북 셀 44 에서 검증).

**기준선**

| | PSNR |
|---|---|
| 기존 USRNet (TTA 없음) | 27.993 |
| 기존 USRNet + TTA 4장 | 28.179 |

이 숫자를 넘어야 v2 가 의미가 있다. **평균만 보지 말고 "몇 장에서 좋아졌는지"도 볼 것** —
val 이 ±0.2 씩 흔들리므로, +0.2 이하의 평균 차이는 운일 수 있지만
100장 중 90장 이상에서 좋아졌다면 그건 진짜다.


In [21]:
FLIP_SETS: dict[str, list[tuple[int, ...]]] = {
    "off": [()],                                 # TTA 없음
    "tta": [(), (-1,), (-2,), (-2, -1)],         # 원본 + 좌우 + 상하 + 둘 다
}


def predict_v2_tta(model: nn.Module, measure: Tensor, dims_list: list[tuple[int, ...]]) -> Tensor:
    """despike -> sigma 추정(원본에서 1회) -> 뒤집어 복원하고 되돌려 평균."""
    mp = despike(measure)
    sigma = estimate_sigma_v2(mp)
    acc: Tensor | None = None
    for dims in dims_list:
        x = torch.flip(mp, dims) if dims else mp
        out = model(x, sigma)
        out = torch.flip(out, dims) if dims else out
        acc = out if acc is None else acc + out
    return finalize(acc / len(dims_list))


# ---- 두 체크포인트 로드 ----
_models: dict[str, nn.Module] = {}
for _tag, _fn in [("best", "best.ckpt"), ("swa", "swa.ckpt")]:
    _p = V2_DIR / _fn
    if not _p.exists():
        print(f"건너뜀: {_p} 없음")
        continue
    _m = USRNetWrapper(pretrained=False)
    _m.load_state_dict(torch.load(str(_p), map_location="cpu", weights_only=True))
    _models[_tag] = _m.to(config.device).eval()
    print(f"loaded: {_p}")

COMBOS = [(f"{_c}_{_t}", _c, _t) for _c in _models for _t in FLIP_SETS]
V2_LABEL = {k: f"v2 {c} + TTA {'4장' if t == 'tta' else '없음'}" for k, c, t in COMBOS}

v2_rows: list[dict] = []
with torch.no_grad():
    for _data in tqdm(test_loader, desc="v2 test", unit="batch"):
        _label = _data[DataKey.Label].to(config.device, non_blocking=True)
        _measure = _data[DataKey.Measure].to(config.device, non_blocking=True)
        _preds = {k: predict_v2_tta(_models[c], _measure, FLIP_SETS[t]) for k, c, t in COMBOS}
        for i, _nm in enumerate(_data[DataKey.Name]):
            _lab_i = _label[i : i + 1]
            _row = {"file": _nm, "noise_type": TEST_NOISE_META.get(_nm, {}).get("noise_type", "unknown")}
            for k, p in _preds.items():
                _row[f"psnr_{k}"] = calculate_psnr(p[i : i + 1], _lab_i).item()
                _row[f"ssim_{k}"] = calculate_ssim(p[i : i + 1], _lab_i).item()
            v2_rows.append(_row)

# ---- 표 ----
_keys = [k for k, _, _ in COMBOS]
print()
print_metric_table(v2_rows, _keys, V2_LABEL)

BASE_OFF, BASE_TTA = 27.993, 28.179
print(f"\n{'방법':<26}{'PSNR':>9}{'기존대비':>11}{'좋아진 장':>11}")
print("-" * 58)
_base_rows = None
_tta_path = config.run_dir / "test_metrics_tta.json"
if _tta_path.exists():
    _prev_rows = {r["file"]: r for r in json.load(open(_tta_path))}
    _base_rows = _prev_rows
for k, _c, _t in COMBOS:
    _v = np.array([r[f"psnr_{k}"] for r in v2_rows])
    _ref = BASE_TTA if _t == "tta" else BASE_OFF
    _win = ""
    if _base_rows:
        _bk = "psnr_tta4" if _t == "tta" else "psnr_tta1"
        _d = np.array([r[f"psnr_{k}"] - _base_rows[r["file"]][_bk] for r in v2_rows])
        _win = f"{int((_d > 0).sum())}/{len(_d)}"
    print(f"{V2_LABEL[k]:<26}{_v.mean():>9.3f}{_v.mean() - _ref:>+11.3f}{_win:>11}")

# ---- 노이즈 종류별 (결과 분석 전용) ----
_types = ["gaussian", "rician", "uniform", "salt_and_pepper"]
print(f"\n{'방법':<26}" + "".join(f"{t[:10]:>13}" for t in _types))
print("-" * (26 + 13 * len(_types)))
print(f"{'기존 USRNet + TTA':<26}" + "".join(f"{v:>13.2f}" for v in [30.88, 22.78, 29.93, 29.12]))
for k, _, _ in COMBOS:
    print(f"{V2_LABEL[k]:<26}" + "".join(
        f"{np.mean([r[f'psnr_{k}'] for r in v2_rows if r['noise_type'] == t]):>13.2f}" for t in _types))

with open(V2_DIR / "test_metrics_v2.json", "w") as f:
    json.dump(v2_rows, f, indent=2)
print(f"\nsaved: {V2_DIR / 'test_metrics_v2.json'}")

loaded: /content/drive/MyDrive/플젝5/logs_final/v2_usrnet/best.ckpt
loaded: /content/drive/MyDrive/플젝5/logs_final/v2_usrnet/swa.ckpt


v2 test:   0%|          | 0/25 [00:00<?, ?batch/s]


method                                n      PSNR       +-      SSIM       +-
-----------------------------------------------------------------------------
v2 best + TTA 없음                    100    28.548    5.482    0.8847   0.1242
v2 best + TTA 4장                    100    28.731    5.487    0.8872   0.1237
v2 swa + TTA 없음                     100    28.553    5.509    0.8844   0.1248
v2 swa + TTA 4장                     100    28.726    5.506    0.8868   0.1243

방법                             PSNR       기존대비      좋아진 장
----------------------------------------------------------
v2 best + TTA 없음             28.548     +0.555     75/100
v2 best + TTA 4장             28.731     +0.552     80/100
v2 swa + TTA 없음              28.553     +0.560     77/100
v2 swa + TTA 4장              28.726     +0.547     81/100

방법                             gaussian       rician      uniform   salt_and_p
------------------------------------------------------------------------------
기존 USRNet + TTA       

## 5. 메모 — 실행 후 채울 것

| 방법 | PSNR | SSIM | 기존 대비 | 좋아진 장 |
|---|---|---|---|---|
| 기존 (TTA 없음) | 27.993 | 0.8697 | — | — |
| 기존 + TTA 4장 | 28.179 | 0.8720 | — | — |
| v2 best (TTA 없음) | 28.548 | 0.8847 | | |
| v2 best + TTA | 28.731 | 0.8872| | |
| v2 swa (TTA 없음) | 28.553 | 0.8844 | | |
| v2 swa + TTA | 28.726 | 0.8868 | | |

**확인할 것**

- [ N ] **SWA 가 best 를 이겼는가?** val 이 ±0.2 흔들리는 상황이라 이겨야 정상이다.
      졌다면 흔들림이 lr 감소만으로 이미 잡혔다는 뜻이라 그것도 정보다.
- [ ] **salt_and_pepper 가 크게 올랐는가?** despike 의 효과는 거의 전부 여기서 나온다.
      사전 측정으로는 입력 노이즈가 약 2.7배 줄었다. 안 올랐으면 despike 가 헛돈 것이다.
- [ ] **gaussian / uniform 이 떨어지지 않았는가?** 사전 예측은 각각 −0.09 / −0.42 였다.
      이보다 크게 떨어졌으면 `DESPIKE_K` 를 12 로 올려 더 조심스럽게 할 것.
- [ ] **rician 은 거의 그대로일 것이다.** despike 도 TTA 도 rician 의 편향은 못 지운다.
      이건 예상된 결과이지 실패가 아니다.
- [ ] **"좋아진 장" 이 90/100 을 넘는가?** 평균 차이가 작아도 이게 크면 진짜 개선이다.

**목표**: 28.179 → **28.7 ~ 29.4**

**여기서 더 올리려면** rician(25장이 22.7점)을 손봐야 하는데,
그건 "이 사진이 magnitude 로 측정됐나"를 판단해야 해서 이번 범위에서 제외했다.
남은 여지는 약 +1.5점이고, 조교/교수님께 허용 여부를 확인한 뒤 판단할 사항이다.


In [ ]:
import sys

if str(USRNET_DIR) not in sys.path:
    sys.path.insert(0, str(USRNET_DIR))
from network_usrnet_v1 import USRNet


class USRNetWrapper(nn.Module):
    def __init__(self, pretrained: bool = True) -> None:
        super().__init__()
        self.net = USRNet(n_iter=8, h_nc=64, in_nc=4, out_nc=3, nc=[64, 128, 256, 512], nb=2,
                          act_mode="R", downsample_mode="strideconv", upsample_mode="convtranspose")
        if pretrained:
            state = torch.load(str(USRNET_DIR / "usrnet.pth"), map_location="cpu", weights_only=True)
            self.net.load_state_dict(state, strict=True)
        self._psf_cache: dict = {}

    def _dipole_psf(self, shape) -> Tensor:
        if shape not in self._psf_cache:
            D = dipole_kernel(shape)
            self._psf_cache[shape] = torch.fft.fftshift(torch.fft.ifft2(D).real)[None, None]
        return self._psf_cache[shape]

    def forward(self, measure: Tensor, sigma: Tensor) -> Tensor:
        x = measure.repeat(1, 3, 1, 1)
        k = self._dipole_psf(tuple(measure.shape[-2:])).to(measure.device)
        out = self.net(x, k, sf=1, sigma=sigma.to(measure.device))
        return out.mean(dim=1, keepdim=True)


CKPT_PATH = None  # 특정 체크포인트를 보려면 여기에 경로 지정 (예: RUN_DIR / "usrnet_r02" / "best.ckpt")

if CKPT_PATH is None:
    _cands = sorted(RUN_DIR.glob("usrnet*/best.ckpt"), key=lambda p: p.stat().st_mtime)
    CKPT_PATH = _cands[-1] if _cands else None

model = USRNetWrapper(pretrained=(CKPT_PATH is None)).to(DEVICE).eval()
if CKPT_PATH is not None:
    model.load_state_dict(torch.load(str(CKPT_PATH), map_location="cpu", weights_only=True))
    print(f"checkpoint: {CKPT_PATH}")
else:
    print("[경고] 학습된 체크포인트가 없어 공식 pretrained 로 복원한다 (fine-tune 안 된 상태)")

: 